# Linear Algebra, Part 1: Hands-On (Applied Math for DS & AI)

Companion notebook for notes 01–05. **Every handwritten example from the lectures (16 Sep – 3 Oct 2026) is verified here**, plus ML connections.

| Part | Note | Topic |
|---|---|---|
| A | 01 | Ax = b: unique / infinite / none; determinant; inverse; least squares |
| B | 01 | "Easy" inverses: identity, reflection, diagonal, orthogonal |
| C | 02 | Gaussian elimination & RREF from scratch; elementary matrices; rank |
| D | 03–04 | Subspace test; linear independence; basis & coordinates |
| E | 05 | Null space, nullity, rank–nullity; **finding redundant features in a dataset** |

In [1]:
import numpy as np
import sympy as sp
from scipy.linalg import null_space
np.set_printoptions(precision=4, suppress=True)

## Part A — Systems of linear equations

In [2]:
# Ex 1 (Lecture 1): unique solution
A = np.array([[2, 1], [1, 2]]); b = np.array([3, 3])
print("det(A) =", np.linalg.det(A).round(4))
print("x = A⁻¹b =", np.linalg.inv(A) @ b, "   np.linalg.solve:", np.linalg.solve(A, b))

det(A) = 3.0
x = A⁻¹b = [1. 1.]    np.linalg.solve: [1. 1.]


In [3]:
# The 2x2 inverse formula from the notes: A⁻¹ = (1/det) [[a22, -a12], [-a21, a11]]
def inv2x2(A):
    (a11, a12), (a21, a22) = A
    det = a11 * a22 - a12 * a21
    if det == 0:
        raise ValueError("det = 0 → A is singular, no inverse")
    return np.array([[a22, -a12], [-a21, a11]]) / det

print(inv2x2(A)); print("A @ A⁻¹ =\n", A @ inv2x2(A))

[[ 0.6667 -0.3333]
 [-0.3333  0.6667]]
A @ A⁻¹ =
 [[1. 0.]
 [0. 1.]]


In [4]:
# Ex 2: infinitely many solutions (2nd equation = 2 × 1st)
A2 = np.array([[2, 1], [4, 2]]); b2 = np.array([3, 6])
print("det =", np.linalg.det(A2), "| rank(A) =", np.linalg.matrix_rank(A2),
      "| rank([A|b]) =", np.linalg.matrix_rank(np.c_[A2, b2]))
for x in [1, 0, 1.5, 2]:
    print(f"x={x}, y={3-2*x}  → check:", A2 @ np.array([x, 3 - 2 * x]))

det = 0.0 | rank(A) = 1 | rank([A|b]) = 1
x=1, y=1  → check: [3 6]
x=0, y=3  → check: [3 6]
x=1.5, y=0.0  → check: [3. 6.]
x=2, y=-1  → check: [3 6]


In [5]:
# Ex 3: no solution (inconsistent) → least-squares 'closest' answer
A3 = np.array([[2, 1], [2, 1]]); b3 = np.array([3, 4])
print("rank(A) =", np.linalg.matrix_rank(A3), "| rank([A|b]) =", np.linalg.matrix_rank(np.c_[A3, b3]), "→ inconsistent")
x_ls, res, rk, _ = np.linalg.lstsq(A3, b3, rcond=None)
print("least-squares x =", x_ls, " → Ax =", A3 @ x_ls, " (splits the difference between 3 and 4)")

rank(A) = 1 | rank([A|b]) = 2 → inconsistent
least-squares x = [1.4 0.7]  → Ax = [3.5 3.5]  (splits the difference between 3 and 4)


💡 Least squares is exactly what **linear regression** does (MLP Note 03): with many noisy equations (one per data point) and few unknowns (weights), there is no exact solution, so we minimise $\|Ax-b\|^2$.

## Part B — Matrices whose inverse is easy

In [6]:
I = np.eye(2)
F = np.array([[0, 1], [1, 0]])                 # reflection about y = x
D = np.diag([4., 0.5])                         # diagonal
print("F @ F =\n", F @ F, "\n→ F is its own inverse")
print("D⁻¹ =\n", np.linalg.inv(D), "\n→ just reciprocals of the diagonal")

# Any reflection about a line through the origin at angle θ is its own inverse
th = np.deg2rad(25)
Ref = np.array([[np.cos(2*th), np.sin(2*th)], [np.sin(2*th), -np.cos(2*th)]])
print("reflection about 25° line, Ref @ Ref =\n", Ref @ Ref)

F @ F =
 [[1 0]
 [0 1]] 
→ F is its own inverse
D⁻¹ =
 [[0.25 0.  ]
 [0.   2.  ]] 
→ just reciprocals of the diagonal
reflection about 25° line, Ref @ Ref =
 [[1. 0.]
 [0. 1.]]


In [7]:
# Anti-diagonal example: [[0, d1], [d2, 0]]
d1, d2 = 3., 5.
Aad = np.array([[0, d1], [d2, 0]])
print("det =", np.linalg.det(Aad), "(= −d1·d2)")
print("inverse =\n", np.linalg.inv(Aad))

det = -15.0 (= −d1·d2)
inverse =
 [[0.     0.2   ]
 [0.3333 0.    ]]


In [8]:
# Orthogonal matrices: A⁻¹ = Aᵀ. Rotation by θ is the classic example.
th = np.deg2rad(30)
R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
print("RᵀR =\n", R.T @ R)
print("column norms:", np.linalg.norm(R, axis=0), "| col1·col2 =", R[:, 0] @ R[:, 1])
v = np.array([3., 4.])
print("‖v‖ =", np.linalg.norm(v), "‖Rv‖ =", np.linalg.norm(R @ v), "→ length preserved")

RᵀR =
 [[1. 0.]
 [0. 1.]]
column norms: [1. 1.] | col1·col2 = 0.0
‖v‖ = 5.0 ‖Rv‖ = 5.0 → length preserved


## Part C — Gaussian elimination, RREF, elementary matrices, rank

In [9]:
def rref(M, tol=1e-12, verbose=False):
    """Reduced row echelon form via Gauss–Jordan elimination (partial pivoting)."""
    A = M.astype(float).copy()
    rows, cols = A.shape
    pivots, r = [], 0
    for c in range(cols):
        if r == rows:
            break
        p = r + np.argmax(np.abs(A[r:, c]))           # best pivot in this column
        if abs(A[p, c]) < tol:
            continue                                    # no pivot here → free column
        if p != r:
            A[[r, p]] = A[[p, r]]                       # 1. row swap
            if verbose: print(f"swap R{r+1} ↔ R{p+1}")
        A[r] = A[r] / A[r, c]                           # 2. scale pivot to 1
        for i in range(rows):
            if i != r and abs(A[i, c]) > tol:
                A[i] -= A[i, c] * A[r]                  # 3. eliminate above AND below
        pivots.append(c); r += 1
    A[np.abs(A) < tol] = 0
    return A, pivots

# Lecture example: x1+x2+x3=3, x1−x2+x3=1, x1+x2−x3=1
Ab = np.array([[1, 1, 1, 3], [1, -1, 1, 1], [1, 1, -1, 1]])
R_, piv = rref(Ab)
print(R_, "\npivot columns:", piv, "→ x1 = x2 = x3 = 1")

[[1. 0. 0. 1.]
 [0. 1. 0. 1.]
 [0. 0. 1. 1.]] 
pivot columns: [0, 1, 2] → x1 = x2 = x3 = 1


In [10]:
# 5×3 example from the notes (REF worked by hand)
A5 = np.array([[1, 1, 1], [1, 1, 1], [1, -1, 1], [1, 1, -1], [-1, 1, -1]])
R5, piv5 = rref(A5)
print(R5, "\nrank =", len(piv5), "| numpy:", np.linalg.matrix_rank(A5))
print("SymPy exact RREF:", sp.Matrix(A5).rref())

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 0.]
 [0. 0. 0.]] 
rank = 3 | numpy: 3
SymPy exact RREF: (Matrix([
[1, 0, 0],
[0, 1, 0],
[0, 0, 1],
[0, 0, 0],
[0, 0, 0]]), (0, 1, 2))


In [11]:
# Elementary row operations ARE matrix multiplications
A = np.array([[2, 1], [1, 2]])
swap  = np.array([[0, 1], [1, 0]])          # R1 ↔ R2
scale = np.array([[4, 0], [0, 1]])          # R1 ← 4 R1
k = 1; shear = np.array([[1, 0], [k, 1]])   # R2 ← R2 + k R1
for name, E in [("swap", swap), ("scale", scale), ("shear", shear)]:
    print(f"{name}: E @ A =\n{E @ A}\n  det(E) = {np.linalg.det(E):.0f}")

swap: E @ A =
[[1 2]
 [2 1]]
  det(E) = -1
scale: E @ A =
[[8 4]
 [1 2]]
  det(E) = 4
shear: E @ A =
[[2 1]
 [3 3]]
  det(E) = 1


In [12]:
# Row ops never change the solution: R2 ← R2 + k R1 for several k
b = np.array([3, 3])
for k in [-2, -1, 0, 1, 2]:
    E = np.array([[1, 0], [k, 1]])
    print(f"k={k:>2}: solution of (EA)x = Eb →", np.linalg.solve(E @ A, E @ b))

k=-2: solution of (EA)x = Eb → [1. 1.]
k=-1: solution of (EA)x = Eb → [1. 1.]
k= 0: solution of (EA)x = Eb → [1. 1.]
k= 1: solution of (EA)x = Eb → [1. 1.]
k= 2: solution of (EA)x = Eb → [1. 1.]


In [13]:
# Rank: three equivalent views (notes p.28)
Ar = np.array([[2, 1], [2, 1]])
print("det =", np.linalg.det(Ar), "→ rank < 2")
print("pivots in RREF:", rref(Ar)[1], "→ rank", len(rref(Ar)[1]))
print("largest non-zero minor: [2] (1×1) → rank 1 | numpy:", np.linalg.matrix_rank(Ar))

det = 0.0 → rank < 2
pivots in RREF: [0] → rank 1
largest non-zero minor: [2] (1×1) → rank 1 | numpy: 1


In [14]:
# Homogeneous system: if x_H solves Ax=0 and x_p solves Ax=b, then x_p + k·x_H solves Ax=b for EVERY k
A = np.array([[2, 1], [4, 2]]); b = np.array([3, 6])
x_p = np.array([1, 1]); x_H = np.array([1, -2])
for k in [0, 1, -3, 10]:
    print(f"k={k:>3}: x = {x_p + k*x_H}, Ax = {A @ (x_p + k*x_H)}")

k=  0: x = [1 1], Ax = [3 6]
k=  1: x = [ 2 -1], Ax = [3 6]
k= -3: x = [-2  7], Ax = [3 6]
k= 10: x = [ 11 -19], Ax = [3 6]


## Part D — Subspaces, linear independence, basis

In [15]:
def is_closed_sample(member, gen, trials=1000, seed=0):
    """Randomised subspace test: contains 0, closed under + and scalar ×."""
    rng = np.random.default_rng(seed)
    if not member(np.zeros(2)):
        return "✗ zero vector missing"
    for _ in range(trials):
        u, v, k = gen(rng), gen(rng), rng.normal() * 10
        if not member(u + v): return f"✗ not closed under +  e.g. {u.round(2)} + {v.round(2)}"
        if not member(k * u): return f"✗ not closed under ×  e.g. {k:.2f}·{u.round(2)}"
    return "✓ passes (consistent with a subspace)"

tests = {
    "S1 = {(x, x)}":        (lambda w: np.isclose(w[0], w[1]),       lambda r: np.full(2, r.normal())),
    "S4 = {(x, 2x)}":       (lambda w: np.isclose(w[1], 2 * w[0]),   lambda r: (lambda t: np.array([t, 2*t]))(r.normal())),
    "S5 = {(x, 3)}":        (lambda w: np.isclose(w[1], 3),          lambda r: np.array([r.normal(), 3.])),
    "lines y=kx, ALL k":    (lambda w: True if np.isclose(w[0], 0) and np.isclose(w[1], 0) else not np.isclose(w[0], 0),
                             lambda r: (lambda t, k: np.array([t, k*t]))(r.normal(), r.normal())),
    "first quadrant x,y≥0": (lambda w: w[0] >= 0 and w[1] >= 0,      lambda r: np.abs(r.normal(size=2))),
}
for name, (m, g) in tests.items():
    print(f"{name:24s} {is_closed_sample(m, g)}")

S1 = {(x, x)}            ✓ passes (consistent with a subspace)


S4 = {(x, 2x)}           ✓ passes (consistent with a subspace)
S5 = {(x, 3)}            ✗ zero vector missing


lines y=kx, ALL k        ✓ passes (consistent with a subspace)
first quadrant x,y≥0     ✗ not closed under ×  e.g. -5.36·[0.13 0.13]


Note the 4th row. The *union* of all lines $y=kx$ passes this sampled test only because it is essentially all of $\mathbb R^2$ minus the vertical axis. Its exact closure fails, e.g. $(1,1)+(-1,1) = (0,2)$ lies on the excluded vertical axis. That's why Ex 6 in the notes must mean a **fixed** $k$. Random testing can *disprove* but never *prove* a subspace; proofs need algebra.

In [16]:
def independent(*vecs):
    M = np.column_stack(vecs)
    return np.linalg.matrix_rank(M) == M.shape[1]

e1, e2, w = np.array([1, 0]), np.array([0, 1]), np.array([2, 3])
print("{e1, e2}      independent?", independent(e1, e2))
print("{e1, e2, w}   independent?", independent(e1, e2, w), "→ 2e1 + 3e2 − w =", 2*e1 + 3*e2 - w)
print("{(1,5),(2,5)} independent?", independent(np.array([1, 5]), np.array([2, 5])))
print("{(1,5),(2,5),(3,5)}?      ", independent(np.array([1, 5]), np.array([2, 5]), np.array([3, 5])),
      "(3 vectors in ℝ² are ALWAYS dependent)")
print("{0}           independent?", independent(np.zeros(2)))

{e1, e2}      independent? True
{e1, e2, w}   independent? False → 2e1 + 3e2 − w = [0 0]
{(1,5),(2,5)} independent? True
{(1,5),(2,5),(3,5)}?       False (3 vectors in ℝ² are ALWAYS dependent)
{0}           independent? False


In [17]:
# Coordinates in a different basis ("spectacles"): same point, different numbers
u, v = np.array([2, 5]), np.array([1, 5])
B = np.column_stack([u, v])
for p in [np.array([1, 0]), np.array([0, 1]), np.array([3, 10]), np.array([-6, 3])]:
    a, b_ = np.linalg.solve(B, p)
    print(f"point {p} = {a:+.2f}·u {b_:+.2f}·v   (standard-basis coords: {p})")

point [1 0] = +1.00·u -1.00·v   (standard-basis coords: [1 0])
point [0 1] = -0.20·u +0.40·v   (standard-basis coords: [0 1])
point [ 3 10] = +1.00·u +1.00·v   (standard-basis coords: [ 3 10])
point [-6  3] = -6.60·u +7.20·v   (standard-basis coords: [-6  3])


In [18]:
# All the bases of ℝ² written in the notes: each has det ≠ 0 and exactly 2 vectors
bases = {"B1": [[1, 0], [0, 1]], "B2": [[2, 1], [1, 2]], "B3": [[1, 4], [-1, 2]], "B4": [[-2, -3], [1, -4]]}
for n, vs in bases.items():
    M = np.column_stack(vs)
    print(n, "det =", round(np.linalg.det(M), 2), "→ basis" if not np.isclose(np.linalg.det(M), 0) else "→ NOT a basis")

B1 det = 1.0 → basis
B2 det = 3.0 → basis
B3 det = 6.0 → basis
B4 det = 11.0 → basis


## Part E — Null space, nullity, rank–nullity

In [19]:
examples = {
    "A = [[1,1],[1,-1],[2,1]]":  np.array([[1, 1], [1, -1], [2, 1]]),
    "A = [[1,1],[1,1],[1,1]]":   np.array([[1, 1], [1, 1], [1, 1]]),
    "A = [[1,1],[1,1]]":         np.array([[1, 1], [1, 1]]),
    "A = [[1,1,2],[1,0,3]]":     np.array([[1, 1, 2], [1, 0, 3]]),
    "A = 0 (5×3)":               np.zeros((5, 3)),
}
for name, A in examples.items():
    m, n = A.shape
    N = sp.Matrix(A).nullspace()                       # exact basis of the null space
    r = np.linalg.matrix_rank(A)
    print(f"{name:26s} n={n}  rank={r}  nullity={len(N)}  rank+nullity={r+len(N)}  "
          f"basis={[list(v) for v in N]}")

A = [[1,1],[1,-1],[2,1]]   n=2  rank=2  nullity=0  rank+nullity=2  basis=[]
A = [[1,1],[1,1],[1,1]]    n=2  rank=1  nullity=1  rank+nullity=2  basis=[[-1, 1]]
A = [[1,1],[1,1]]          n=2  rank=1  nullity=1  rank+nullity=2  basis=[[-1, 1]]
A = [[1,1,2],[1,0,3]]      n=3  rank=2  nullity=1  rank+nullity=3  basis=[[-3, 1, 1]]
A = 0 (5×3)                n=3  rank=0  nullity=3  rank+nullity=3  basis=[[1, 0, 0], [0, 1, 0], [0, 0, 1]]


Every row satisfies **rank + nullity = n (number of columns)**: the *rank–nullity theorem*. You'll prove it in class soon.

In [20]:
# The 2×3 example from 3 Oct, step by step
A = sp.Matrix([[1, 1, 2], [1, 0, 3]])
print("RREF:", A.rref())
x1, x2, x3, t = sp.symbols("x1 x2 x3 t")
print("solution:", sp.solve(A * sp.Matrix([x1, x2, x3]), [x1, x2]), "→ x = t·(−3, 1, 1)")
print("check A·(−3,1,1) =", list(A * sp.Matrix([-3, 1, 1])))
print("i.e. −3·col1 + col2 + col3 = 0  → the columns are dependent")

RREF: (Matrix([
[1, 0,  3],
[0, 1, -1]]), (0, 1))


solution: {x1: -3*x3, x2: x3} → x = t·(−3, 1, 1)
check A·(−3,1,1) = [0, 0]
i.e. −3·col1 + col2 + col3 = 0  → the columns are dependent


### E2. ML application: detecting redundant features
A dataset is a matrix (rows = observations, columns = variables, as in Lecture 1). If some columns are linear combinations of others, the **nullity** of the data matrix counts those redundant directions.

In [21]:
rng = np.random.default_rng(42)
n = 1000
height_cm = rng.normal(165, 10, n)
weight_kg = rng.normal(65, 12, n)
age_yr    = rng.integers(18, 60, n).astype(float)
X = np.column_stack([
    height_cm,
    height_cm / 2.54,                    # height in inches: redundant (scaled copy)
    weight_kg,
    age_yr,
    2025 - age_yr,                       # birth year: affine, see note below
    0.5 * height_cm + 2 * weight_kg,     # an engineered combination: redundant
])
names = ["height_cm", "height_in", "weight", "age", "birth_year", "0.5h+2w"]
r = np.linalg.matrix_rank(X)
print(f"shape {X.shape}, rank {r}, nullity {X.shape[1] - r}")
# RREF of the data matrix: non-pivot columns are combinations of the pivot columns
R, piv = rref(X, tol=1e-8)
print("independent (pivot) columns:", [names[i] for i in piv])
for j in range(X.shape[1]):
    if j not in piv:
        terms = " ".join(f"{R[i, j]:+.4f}·{names[c]}" for i, c in enumerate(piv) if abs(R[i, j]) > 1e-8)
        print(f"redundant: {names[j]} = {terms}")

shape (1000, 6), rank 4, nullity 2
independent (pivot) columns: ['height_cm', 'weight', 'age', 'birth_year']
redundant: height_in = +0.3937·height_cm
redundant: 0.5h+2w = +0.5000·height_cm +2.0000·weight


**Reading the output:** nullity 2 → two independent linear dependencies (inches ↔ cm; the engineered column ↔ height & weight). `birth_year = 2025 − age` is **affine** (has a constant). It only becomes linearly dependent once you add a column of ones (the intercept/bias, as in regression's $x_0=1$). Add it and the nullity becomes 3.

As the professor said, nullity tells you **how many** columns are redundant, but **not which ones**: you could drop `height_cm` *or* `height_in`.

In [22]:
X1 = np.column_stack([np.ones(n), X])
print("with intercept column: rank", np.linalg.matrix_rank(X1), "of", X1.shape[1], "→ nullity", X1.shape[1] - np.linalg.matrix_rank(X1))

# Consequence for regression: XᵀX becomes singular → the normal equation breaks
print("cond(XᵀX) =", f"{np.linalg.cond(X1.T @ X1):.2e}", " (huge = (near-)singular, unstable inverse)")

with intercept column: rank 4 of 7 → nullity 3
cond(XᵀX) = 1.14e+20  (huge = (near-)singular, unstable inverse)
